# Geographic Mapping and Marine Data Superimposition: Aruba

This notebook provides a detailed comparison and implementation of two fundamental geospatial visualization approaches in Python:
1. **Static Publication-Ready Mapping** using **GeoPandas**, **Xarray**, and **Matplotlib**.
2. **Interactive Web-Ready Mapping** using **GeoJSON**, **Folium**, and **Raster ImageOverlays**.

### The Scientific Context
When working with sustainable energy, atmospheric, or marine systems in Small Island Developing States (SIDS) like Aruba, visual overlays of spatial grids (e.g., wind speeds, currents, sea surface temperature, or pH) on top of island boundaries are critical. We use:
- `aruba.geojson`: Vector boundary of the main island of Aruba.
- `aruba_phy.nc`: Copernicus Marine Service (CMEMS) physical NetCDF data containing sea surface temperature (`thetao`), salinity (`so`), and current velocities (`uo`, `vo`).

In [ ]:
# Install dependencies if needed:
# !pip install geopandas folium xarray pillow branca pyogrio

import xarray as xr
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.raster_layers import ImageOverlay
import branca.colormap as cm
from PIL import Image

print("All libraries imported successfully!")

## Section 1: The Vector Foundation - Loading Aruba's Boundary with GeoPandas

We load the GeoJSON boundary of Aruba using GeoPandas, which represents vector shape data in a GeoDataFrame, preserving coordinate reference systems (CRS).

In [ ]:
# Load the Aruba GeoJSON boundary
gdf = gpd.read_file('aruba.geojson')

print("--- GeoDataFrame Information ---")
print(f"Type: {type(gdf)}")
print(f"Shape: {gdf.shape} (Features, Columns)")
print(f"Coordinate Reference System (CRS): {gdf.crs}")
print("\n--- Properties and Schema ---")
display(gdf.head())

# Quick static plot of the vector boundary alone
fig, ax = plt.subplots(figsize=(6, 6))
gdf.plot(ax=ax, facecolor='#1abc9c', edgecolor='#2c3e50', linewidth=1.5, alpha=0.3)
ax.set_title("Aruba Boundary Vector Layer (EPSG:4326)", fontsize=12, fontweight='bold')
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Exporting Vector Layers to GeoPackage (.gpkg) for QGIS

While GeoJSON is a great text-based format for web applications, the **GeoPackage (.gpkg)** format is the industry standard for GIS platforms like **QGIS**. 
GeoPackage is an open, standards-based, SQLite-database container. It is highly optimized, supports multiple spatial layers in a single file, handles indexing, and loads instantly in QGIS.

We can save our GeoDataFrame to a GeoPackage layer with a single line of code:

In [ ]:
# Export the GeoDataFrame as a layer in a GeoPackage
gpkg_filename = 'aruba_osm_highres.gpkg'
layer_name = 'aruba_coastline'

print(f"Writing GeoDataFrame to {gpkg_filename} as layer '{layer_name}'...")
gdf.to_file(gpkg_filename, driver='GPKG', layer=layer_name)
print("Export completed! You can now drag and drop 'aruba.gpkg' directly into QGIS.")

## Section 2: Static Mapping with GeoPandas and Xarray (Publication-Ready)

In scientific papers and reports, static maps are preferred for their stability, high resolution, and precise vector overlay (like quiver arrows representing physical fluid currents).

Here, we read the physical marine data `aruba_phy.nc` (Sea Surface Temperature and Current Velocity) using `xarray`, extract a surface slice, and render a multi-layered static visualization.

In [ ]:
# Load the physical NetCDF dataset
ds = xr.open_dataset('aruba_phy.nc')
display(ds)

# Extract a single slice: first time step, surface depth level
time_val = ds['time'].values[0]
depth_val = ds['depth'].values[0]
print(f"\nSlicing Dataset at:\n- Time: {time_val}\n- Depth: {depth_val:.2f} meters")

ds_slice = ds.sel(time=time_val, depth=depth_val)

# Extract coordinate and variable arrays
lons = ds_slice['longitude'].values
lats = ds_slice['latitude'].values
sst = ds_slice['thetao'].values     # Sea Water Temperature in °C
uo = ds_slice['uo'].values          # Eastward current velocity in m/s
vo = ds_slice['vo'].values          # Northward current velocity in m/s

# ----------------- PLOTTING -----------------
fig, ax = plt.subplots(figsize=(10, 8))

# 1. Plot temperature as a continuous raster background mesh
mesh = ax.pcolormesh(lons, lats, sst, cmap='coolwarm', shading='auto', alpha=0.85)
cbar = fig.colorbar(mesh, ax=ax, label='Sea Surface Temperature (°C)', pad=0.02)
cbar.ax.tick_params(labelsize=10)

# 2. Overlay ocean currents as vector arrows (quiver plot)
lon_grid, lat_grid = np.meshgrid(lons, lats)
quiv = ax.quiver(lon_grid, lat_grid, uo, vo, color='#2c3e50', 
                 scale=6, width=0.0035, headwidth=4, headlength=5, alpha=0.8,
                 label='Current Velocity (m/s)')
ax.quiverkey(quiv, 0.82, 0.05, 0.2, label='0.2 m/s Current', labelpos='E', coordinates='figure', fontproperties={'weight': 'bold'})

# 3. Superimpose the GeoPandas vector boundary of Aruba on top
gdf.plot(ax=ax, facecolor='none', edgecolor='#111111', linewidth=2, label='Aruba Boundary')

# Style the map elements
ax.set_title(f"Aruba Marine Climatology: SST & Ocean Currents\nDepth: {depth_val:.2f}m | Time: {np.datetime_as_string(time_val, unit='h')}", 
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Longitude (Degrees West)", fontsize=11)
ax.set_ylabel("Latitude (Degrees North)", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xlim(-70.22, -69.62)
ax.set_ylim(12.28, 12.72)

plt.tight_layout()
plt.show()

## Section 3: Interactive Mapping with GeoJSON and Folium (Web-Ready)

For interactive dashboards, web portals, or field exploration, an interactive map allows users to zoom, pan, query telemetry, and toggle layers.

Folium integrates Leaflet.js map layers into Python. We will build a dark-themed interactive map, add the Aruba boundary from `aruba.geojson` as a styled GeoJSON layer, and superimpose the Sea Surface Temperature grid as a raster `ImageOverlay`.

In [ ]:
# Center of Aruba
aruba_center = [12.518, -69.967]

# Initialize Leaflet Map with dark tiles for rich premium aesthetics
m = folium.Map(location=aruba_center, zoom_start=11, tiles="CartoDB dark_matter")

# Add the Aruba GeoJSON boundary with custom premium glassmorphism styling
folium.GeoJson(
    'aruba.geojson',
    name='Aruba Land Boundary',
    style_function=lambda x: {
        'fillColor': '#1abc9c',   # Translucent teal
        'color': '#2ecc71',       # Emerald green outline
        'weight': 3,
        'fillOpacity': 0.12
    },
    tooltip=folium.Tooltip("Aruba Boundary (Vector Layer)")
).add_to(m)

m

### Superimposing Gridded Marine Telemetry on Folium

To overlay the temperature grid on the interactive map, we convert the 2D xarray data slice into a colormapped RGBA PNG. We make the land masses (NaN values) transparent, flip the image vertically to match the top-down image coordinates to bottom-up geographic coordinates, and add it using Folium's `ImageOverlay`.

In [ ]:
# Get temperature range
sst_min, sst_max = float(np.nanmin(sst)), float(np.nanmax(sst))

# Normalize SST to [0, 1] range for colormapping
norm_sst = (sst - sst_min) / (sst_max - sst_min)

# Map normalized values to RGBA using matplotlib's colormap
cmap = plt.get_cmap('coolwarm')  # Match the static plot's colormap
rgba_img = cmap(norm_sst)

# Set NaN values (e.g. land, out of bounds) to be completely transparent
rgba_img[np.isnan(sst)] = [0.0, 0.0, 0.0, 0.0]

# Convert to 8-bit unsigned integer array (0-255)
rgba_img_8bit = (rgba_img * 255).astype(np.uint8)

# Flip image vertically because image array index starts from top-left, 
# while geographic coordinates (lats) go bottom-to-top (ascending)
if lats[0] < lats[-1]:
    rgba_img_8bit = np.flipud(rgba_img_8bit)

# Save the RGBA array as a PNG using Pillow
overlay_img_path = 'scratch/sst_overlay.png'
pil_img = Image.fromarray(rgba_img_8bit)
pil_img.save(overlay_img_path)
print(f"Generated transparent raster overlay: {overlay_img_path}")

# Geographic bounds for ImageOverlay: [[min_lat, min_lon], [max_lat, max_lon]]
min_lat, max_lat = float(lats.min()), float(lats.max())
min_lon, max_lon = float(lons.min()), float(lons.max())
bounds = [[min_lat, min_lon], [max_lat, max_lon]]

# Add the raster ImageOverlay layer to the map
ImageOverlay(
    image=overlay_img_path,
    bounds=bounds,
    opacity=0.6,
    name='Sea Surface Temperature (°C)',
    interactive=True,
    cross_origin=False
).add_to(m)

# Create and add a coolwarm-like colorbar using branca's LinearColormap
coolwarm_colors = ['#3b4cc0', '#8db0fe', '#f2f2f2', '#f49a7a', '#b40426']
colorbar_legend = cm.LinearColormap(
    colors=coolwarm_colors,
    vmin=sst_min,
    vmax=sst_max,
    caption='Sea Surface Temperature (°C)'
)
colorbar_legend.add_to(m)

# Add Layer Control to toggle layers (Land boundary vs. Temperature overlay)
folium.LayerControl().add_to(m)

# Display the interactive map
m

## Section 4: Quantitative Comparison and Tradeoffs

| Feature / Criteria | GeoPandas + Matplotlib (Static Mapping) | GeoJSON + Folium (Interactive Mapping) |
| :--- | :--- | :--- |
| **Output Format** | Static Image (PNG, PDF, SVG) | Interactive HTML/JS Webpage |
| **Interactivity** | None (Static presentation) | Zooming, Panning, Layer Toggles, Tooltips, Popups |
| **Current Velocities (Quiver)** | Native and excellent via `ax.quiver()` vector arrows | Requires custom Canvas/WebGL plugins, or discrete arrow markers |
| **Performance (Large Grid)** | High; rendered server-side as static pixels | Moderate; rendering millions of interactive elements/GeoJSON features lags the browser |
| **Colorbar Legends** | Rich and flexible native colorbars via Matplotlib | Standard web-based colorbar via `branca.colormap` |
| **Primary Use Cases** | Scientific publications, theses, formal reports, print layouts | Web dashboards, client presentations, interactive data exploration |

### Extending to Biogeochemical Marine/Atmospheric Data
To visualize other datasets in the workspace, you can apply these same steps. For example:
- Load `aruba_bgc.nc` or `CMVarubaph.nc` to visualize **pH**, **Dissolved Inorganic Carbon (`dissic`)**, or **Total Alkalinity (`talk`)**.
- Load custom wind data arrays (e.g. Weibull speed profiles) to simulate wind turbine sites.

For these variables, simply swap out the variable name in `ds_slice['thetao']` to `ds_slice['ph']` and update the colormaps accordingly (e.g., using `viridis` or `plasma`).

## Section 5: Bulk SIDS GeoPackage Creator

For studies involving multiple Small Island Developing States (SIDS), we can automate the boundary fetching process. 
Below is a bulk downloading block that iterates over all **58 SIDS countries and territories** (as categorized by UN-OHRLLS), queries Nominatim for their high-resolution vector shapes, and exports each as a separate GeoPackage layer inside a local `sidsgeopackages/` directory.

**Rate Limiting Warning**: Nominatim requests a maximum of **1 request per second**. The code includes a `time.sleep(1.5)` between iterations to comply with this policy and prevent IP blocks.

In [ ]:
import os
import time
import requests
import geopandas as gpd

# Complete list of SIDS entities
sids_list = [
    # Caribbean Sovereign
    "Antigua and Barbuda", "Bahamas", "Barbados", "Belize", "Cuba", 
    "Dominica", "Dominican Republic", "Grenada", "Guyana", "Haiti", 
    "Jamaica", "Saint Kitts and Nevis", "Saint Lucia", 
    "Saint Vincent and the Grenadines", "Suriname", "Trinidad and Tobago",
    # Caribbean Territories
    "Anguilla", "Aruba", "Bermuda", "British Virgin Islands", 
    "Cayman Islands", "Curaçao", "Guadeloupe", "Martinique", 
    "Montserrat", "Puerto Rico", "Sint Maarten", 
    "Turks and Caicos Islands", "U.S. Virgin Islands",
    # Pacific Sovereign
    "Fiji", "Kiribati", "Marshall Islands", "Federated States of Micronesia", "Nauru", 
    "Palau", "Papua New Guinea", "Samoa", "Solomon Islands", 
    "Timor-Leste", "Tonga", "Tuvalu", "Vanuatu",
    # Pacific Territories
    "American Samoa", "Northern Mariana Islands", "Cook Islands", 
    "French Polynesia", "Guam", "New Caledonia", "Niue",
    # AIS Sovereign
    "Bahrain", "Cabo Verde", "Comoros", "Guinea-Bissau", "Maldives", 
    "Mauritius", "São Tomé and Príncipe", "Seychelles", "Singapore"
]

def sanitize_name(name):
    clean = name.lower().replace(" ", "_").replace(".", "").replace(",", "").replace("-", "_")
    clean = clean.replace("ç", "c").replace("ã", "a").replace("é", "e").replace("í", "i").replace("ô", "o")
    return clean

output_folder = 'sidsgeopackages'
os.makedirs(output_folder, exist_ok=True)
headers = {"User-Agent": "SIDSBulkDownloader/1.0 (Sustainable Siting Research)"}

print(f"Initiating download of {len(sids_list)} SIDS boundaries to '{output_folder}/'...")

for i, entity in enumerate(sids_list, 1):
    filename = f"{sanitize_name(entity)}_osm_highres.gpkg"
    gpkg_path = os.path.join(output_folder, filename)
    
    if os.path.exists(gpkg_path):
        print(f"[{i}/{len(sids_list)}] {entity} already exists. Skipping.")
        continue
        
    print(f"[{i}/{len(sids_list)}] Querying boundary for: {entity}...")
    url = f"https://nominatim.openstreetmap.org/search?q={requests.utils.quote(entity)}&format=geojson&polygon_geojson=1"
    
    try:
        # Sleep to comply with Nominatim rate limits (1 request/second)
        time.sleep(1.5)
        
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            data = r.json()
            features = data.get('features', [])
            
            if not features:
                print(f"  --> Warning: No geometry returned for {entity}")
                continue
                
            # Filter to select MultiPolygon or Polygon
            selected = None
            for f in features:
                if f.get('geometry', {}).get('type') == 'MultiPolygon':
                    selected = f
                    break
            if not selected:
                for f in features:
                    if f.get('geometry', {}).get('type') == 'Polygon':
                        selected = f
                        break
            if not selected:
                selected = features[0]
                
            clean_geojson = {'type': 'FeatureCollection', 'features': [selected]}
            gdf_entity = gpd.GeoDataFrame.from_features(clean_geojson)
            if gdf_entity.crs is None:
                gdf_entity.set_crs('EPSG:4326', inplace=True)
                
            layer_name = f"{sanitize_name(entity)}_boundary"
            gdf_entity.to_file(gpkg_path, driver='GPKG', layer=layer_name)
            print(f"  --> Success! Saved layer '{layer_name}' to {gpkg_path}")
        else:
            print(f"  --> HTTP Error {r.status_code} for {entity}")
    except Exception as e:
        print(f"  --> Error processing {entity}: {e}")

print("\nBulk downloading process completed!")